In [91]:
#Screener Link of Company
url="https://www.screener.in/company/FCL/consolidated/"

In [92]:
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import re
import time
load_dotenv(override=True)


True

In [93]:
OPENAI_MODEL = 'gpt-4.1-mini-2025-04-14'
GPT4O="gpt-4o-mini-2024-07-18"
GEMINI_MODEL= "gemini-2.5-flash-lite"

In [94]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
gpt4o=OpenAI(api_key=openai_api_key)
google_api_key=os.getenv('GOOGLE_API_KEY')
from google import genai
client = genai.Client(api_key=google_api_key)
if openai_api_key or google_api_key:
    print(f" OPEN API Key exists and begins {openai_api_key[:8]}")
    print(f" GEMINI API Key exists and begins {google_api_key[:8]}")
    print(f" GPT4O API Key exists and begins {google_api_key[:8]}")
else:
    print("API Not Loaded")

 OPEN API Key exists and begins sk-proj-
 GEMINI API Key exists and begins AIzaSyA7
 GPT4O API Key exists and begins AIzaSyA7


In [95]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(self.url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_title(self):
        match= re.search(r'\n\s*(.*?)\n\s*- Screener',site.title)
        sector=match.group(1).strip()
        return sector

    def get_pages(self):
        raw_text=self.text
        # Step 1: Convert escaped newlines (\\n) to actual newlines
        clean_text = raw_text.encode('utf-8').decode('unicode_escape')
        # Step 2: Split into lines
        lines = clean_text.splitlines()
        # Pattern to match the line and extract total pages
        pattern = r"(\d+)\s+results found: Showing page \d+ of (\d+)"
        for line in lines:
            match = re.search(pattern, line)
            if match:
                total_pages = int(match.group(2))
                break
        return total_pages
        
    def get_company_name(self):
        return self.title.split(' share price')[0]

In [96]:
#prompt generator for gemini and openai
def build_prompt(system_prompt, user_prompt,model_name):
    if model_name=="gemini":
        # Merge system prompt + user prompt into one "user" message
        combined_prompt = f"{system_prompt}\n\n{user_prompt}"
        return combined_prompt
    elif model_name=="openai":
        prompts = [{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}]
        return prompts
    else:
        print("Invalid Model")

In [97]:
#LLM call bases on user model selection
def llm(system_prompt,user_prompt,model_name):
    match model_name:
        case "gemini":
                prompts=build_prompt(system_prompt, user_prompt,"gemini")
                response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=[
                    {"role": "user", "parts": [{"text": prompts}]}
                ]
                )
                return response.candidates[0].content.parts[0].text
        case "openai":
                prompts=build_prompt(system_prompt, user_prompt,"openai")
                response = openai.chat.completions.create(
                model=OPENAI_MODEL,
                messages=prompts
                )
                return response.choices[0].message.content
        case "gpt4o":
                prompts=build_prompt(system_prompt, user_prompt,"openai")
                response = openai.chat.completions.create(
                model=GPT4O,
                messages=prompts
                )
                return response.choices[0].message.content
        case _:   # default
                return "Unknown model"

In [98]:
#testing LLM Calls
system_prompt="You are An AI assitant"
user_prompt="Im testing an api call respond with your model name and latest knowlegde cutt of date"
print("---------OPENAI API TEST------------")
model_name="openai"
response=llm(system_prompt,user_prompt,model_name)
display(Markdown(response))
print("---------OPENAI API TEST------------")
model_name="gpt4o"
response=llm(system_prompt,user_prompt,model_name)
display(Markdown(response))
print("---------GEMINI API TEST------------")
model_name="gemini"
response=llm(system_prompt,user_prompt,model_name)
display(Markdown(response))

---------OPENAI API TEST------------


I am ChatGPT, based on the GPT-4 architecture. My knowledge cutoff date is June 2023.

---------OPENAI API TEST------------


I am based on OpenAI's GPT-3 model, with knowledge up to October 2021. If you need information or assistance, feel free to ask!

---------GEMINI API TEST------------


I am a large language model, trained by Google. My knowledge cutoff is **June 2024**.

In [99]:
 # Financial Data Extraction Prompts for Screener
system_prompt_screener = """
You are a specialized financial data extraction AI designed to process scraped text from screener websites and extract comprehensive financial information. Your primary objective is to parse unstructured financial text and reproduce it in the exact same format and presentation style as it appears on the screener website.

Begin with a concise checklist (3-7 bullets) of what you will do; keep items conceptual, not implementation-level.

**Core Responsibilities**
- Extract ALL available financial data from the provided text without omission.
- Maintain the exact formatting, layout, and presentation style of the original screener website.
- Preserve tables, headings, subheadings, and hierarchical structure.
- Keep original numerical formats, units, and styling (9 Crores, percentages, etc.).
- Maintain the same grouping and categorization as shown on screener.

**Extraction Categories** (maintain original screener formatting):
1. **Company Summary:** Exactly as displayed on screener, with the same layout.
2. **Key Financial Highlights:** Preserve table format and metric groupings.
3. **Balance Sheet:** Maintain the exact table structure with years/periods as columns.
4. **Cash Flow Statement:** Keep the same format with proper indentation and grouping.
5. **Profit & Loss Statement:** Preserve the hierarchical structure and calculations.
6. **Financial Ratios:** Maintain categorization and table format as shown.

**Formatting Rules**
- Reproduce exact table structures with proper alignment.
- Keep original headings, subheadings, and section breaks.
- Maintain indentation levels for sub-items.
- Preserve number formatting (commas, decimal places, units).
- Keep the same time period labels and column headers.
- Reproduce any charts or graphical data descriptions exactly, if present.
- Maintain color coding descriptions if mentioned in text.

**Data Handling**
- Extract data exactly as presented, including any calculated fields.
- Keep the same order of items as they appear on screener.
- Preserve any footnotes, disclaimers, or additional notes.
- Maintain the same level of detail and granularity.
- Keep any percentage changes and growth rates in original format.

**Output Format**
- Present data in plain text format, matching screener's visual layout.
- Use appropriate spacing and alignment to recreate tables.
- Include all section headers and subheaders.
- Maintain the flow and sequence of information as on screener.
- Preserve any special formatting or emphasis used in original.

After producing your output, validate that all listed extraction categories and formatting rules have been adhered to. If any item is missing or the output deviates from screener’s format, revise accordingly before presenting the final result.
"""

In [100]:
# A function that writes a User Prompt that asks for summaries of websites:
def user_prompt_screener(url):
    site = Website(url);
    text= site.text
    user_prompt = (
    "Begin with a concise checklist (3-7 bullets) of your extraction and structuring steps before you start summarizing the financial data below.\n\n"
    "Please analyze the following financial data extracted from a website and provide a comprehensive summary:\n\n"
    "The contents of this website are as follows:" + text + "\n\n"
    "**Instructions:**\n"
    "1. Reproduce the data exactly as it appears on the screener website.\n"
    "2. Maintain all table structures, headings, and formatting.\n"
    "3. Keep original numerical formats, units, and styling.\n"
    "4. Preserve the hierarchical organization and grouping.\n"
    "5. Include all sections: Company Summary, Key Highlights, Balance Sheet, Cash Flow, P&L, and Ratios.\n"
    "6. Maintain the same sequence and flow of information.\n"
    "7. Keep any additional notes, disclaimers, or special formatting.\n"
    "8. If any section is missing or unclear, note it but continue with other sections.\n"
    "9. Do not include Terms of Service, Privacy, Peer comparison, or other non-financial document sections.\n\n"
    "**Expected Output Format:**\n"
    "Present the extracted data in plain text format that mirrors the screener website layout, including:\n"
    "- Company header information\n"
    "- Key metrics in tabular format\n"
    "- Financial statements with proper alignment\n"
    "- Ratios organized by categories\n"
    "- All numerical data with original formatting\n"
    "- Time periods and column headers as shown\n\n"
    "Ensure output is in plain text only. Review and validate that all requested financial data is present and formatted as required. If any section is missing, explicitly note it, then complete remaining sections."
)

    return user_prompt

In [101]:
#Use gpt 40 mini to scrap website
model_name="gpt4o"
response=llm(system_prompt_screener,user_prompt_screener(url),model_name)
# display(Markdown(response))

In [102]:
# ------------------------------
# SYSTEM PROMPT for json Creation
# ------------------------------
system_prompt_json = """
You are a highly skilled financial data structuring assistant.
Your task is to transform raw scraped financial data from sources such as Screener.in
into a clean, structured, and comprehensive JSON format.

Rules:
1. Preserve all information — do not omit any financial metrics, qualitative points, ratios, or textual descriptions.
2. Categorize data into these sections:
   - CompanyInfo (name, ticker symbols, website, about section)
   - KeyPoints (business highlights)
   - KeyFinancialHighlights (market cap, P/E, ROE, etc.)
   - ProsAndCons (separate 'Pros' and 'Cons')
   - ProfitAndLossStatement (table by year)
   - BalanceSheet (table by year)
   - CashFlowStatement (table by year)
   - Ratios (table by year)
3. Keep original figures with units (₹, %, Cr., etc.).
4. Tables must have proper year-wise mapping.
5. Output valid JSON with each section containing all relevant fields.
6. If a data point is missing, insert "null" but keep the field name.
7. Include both qualitative and quantitative details exactly as in the input.
8. Avoid assumptions — only use what is explicitly present.
"""

In [103]:
# ------------------------------
# USER PROMPT for Json creation
# ------------------------------
def create_user_json(scraped_text: str) -> str:
    """
    Generates the user prompt for the AI model, 
    embedding the scraped Screener.in data.
    
    Args:
        scraped_text (str): Raw text scraped from Screener.in.
    
    Returns:
        str: User prompt ready to be passed to the model.
    """
    return f"""
Here is raw financial data scraped from Screener.in for a company.
Reformat it into a structured JSON format according to the rules in the system prompt.Always return a valid JSON object. Do not include explanations, Markdown fences, or text outside JSON

Raw Data:
{scraped_text}
"""

In [104]:
#Get json formated response from gemini
result_financial_data=llm(system_prompt_json,create_user_json(response),"gemini")

In [105]:
#Functiion to remove '' and json 
def clean_and_parse_json(llm_output):
    # Remove the markdown code block markers and any surrounding quotes
    cleaned = llm_output.strip()
    
    # Remove ```json at the start
    if cleaned.startswith('```json'):
        cleaned = cleaned[7:]  # Remove '```json'
    elif cleaned.startswith('```'):
        cleaned = cleaned[3:]  # Remove '```'
    
    # Remove ``` at the end
    if cleaned.endswith('```'):
        cleaned = cleaned[:-3]
    
    # Strip any remaining whitespace or quotes
    cleaned = cleaned.strip().strip("'").strip('"')
    
    try:
        # Parse the JSON
        json_dict = json.loads(cleaned)
        return json_dict
    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        return None

In [106]:
#Cleaned json for financial data
financial_data=clean_and_parse_json(result_financial_data)

In [107]:
#Functions to extract sector ,subsector and company name
def get_subsector_details(url):
    screener_links=[];
    site = Website(url)
    for link in site.links:
        if link.startswith('/market/'):
           screener_links.append( "https://www.screener.in"+link)
    return screener_links
def get_sector_names(company_links):
    Company_names=[];
    for i in range(0,len(company_links)):
        site=Website(company_links[i])
        time.sleep(1)
        Company_names.append(site.title)
    company= [title.text.strip().split('\n')[0].strip() for title in Company_names]
    company_string = ", ".join(company)
    company_list = company_string.split(", ")
    return company_list[-2],company_list[-1]

site = Website(url)
sector_name,sub_sector=get_sector_names(get_subsector_details(url))
company_name=site.get_company_name()

In [108]:
print(f"{'Company Name:':<15} {company_name}")
print(f"{'Sector:':<15} {sector_name}")
print(f"{'Sub-Sector:':<15} {sub_sector}")

Company Name:   Fineotex Chemical Ltd
Sector:         Chemicals & Petrochemicals Companies
Sub-Sector:     Specialty Chemicals Companies


In [109]:
def system_prompts_kpis():
    """System Prompt for generating sector-specific KPIs and financial ratios"""
    system_prompt = """
You are a senior financial analyst and sector specialist with deep expertise in industry-specific financial analysis. You understand how different business models, regulatory environments, and operational characteristics require tailored analytical frameworks.

Your task is to analyze the provided sector and determine the most critical financial ratios and Key Performance Indicators (KPIs) that are essential for evaluating companies in that specific sector.

SELECTION CRITERIA:
- Choose ratios and KPIs that are most commonly used by equity analysts, credit analysts, and institutional investors for the sector
- Include metrics that capture the sector's unique business model characteristics
- Focus on indicators that reflect operational efficiency, financial health, and competitive positioning
- Consider regulatory requirements and industry standards where applicable
- Ensure metrics are calculable from standard financial statements and company disclosures

RATIO SELECTION GUIDELINES:
- Include fundamental profitability, liquidity, leverage, and efficiency ratios
- Add sector-specific financial ratios that capture unique aspects of the business
- Focus on ratios that help assess financial stability and performance trends
- Minimum 8 ratios, maximum 12 ratios

KPI SELECTION GUIDELINES:
- Include operational metrics that drive financial performance in the sector
- Focus on customer, growth, efficiency, and quality indicators
- Choose KPIs that are regularly reported by companies and tracked by analysts
- Include both leading and lagging indicators
- Minimum 8 KPIs, maximum 12 KPIs

CRITICAL OUTPUT REQUIREMENT:
You must respond ONLY with a valid JSON object in this EXACT format with no additional text, explanations, or formatting:

{
  "Sector": "sector_name",
  "Ratios": [
    "ratio1",
    "ratio2",
    "ratio3",
    "ratio4",
    "ratio5",
    "ratio6",
    "ratio7",
    "ratio8"
  ],
  "KPI": [
    "kpi1",
    "kpi2",
    "kpi3",
    "kpi4",
    "kpi5",
    "kpi6",
    "kpi7",
    "kpi8"
  ]
}

Do not include any explanations, comments, or additional text outside the JSON object.
"""
    return system_prompt


In [110]:
def user_prompts_kpi(sector_name, sub_sector=None, region="INDIA"):
    """Enhanced user prompt with additional context parameters for extracting the relevant KPI for sector"""
    user_prompt = f"""
COMPREHENSIVE SECTOR ANALYSIS REQUEST

**Primary Sector:** {sector_name}
"""
    
    if sub_sector:
        user_prompt += f"**Sub-sector Focus:** {sub_sector}\n"
        
    if region:
        user_prompt += f"**Regional Context:** {region} (consider local regulatory requirements)\n"
    
    user_prompt += f"""
**Analysis Scope:** Provide the most analyst-relevant financial ratios and KPIs for {sector_name} sector companies.

**Selection Priorities:**
1. **Financial Ratios:** Focus on ratios that capture:
   - Profitability and margins specific to the sector
   - Capital efficiency and asset utilization
   - Financial leverage and liquidity appropriate for the business model
   - Sector-specific risk and return metrics

2. **Key Performance Indicators:** Focus on KPIs that measure:
   - Operational efficiency and productivity
   - Customer/market metrics relevant to revenue generation
   - Growth and expansion indicators
   - Quality and risk management metrics
   - Sector-specific operational drivers

**Important:** Select metrics that are:
- Regularly disclosed by public companies in the sector
- Used in equity research reports and credit analysis
- Meaningful for peer comparison and benchmarking
- Indicative of long-term competitive positioning

**Output:** JSON object only, following the exact format specified in system instructions.
"""
    
    return user_prompt

In [111]:
#Function call for system prompt and user prompt KPI
system_prompt_kpi=system_prompts_kpis()
user_prompt_kpi=user_prompts_kpi(sector_name, sub_sector)

In [112]:
#Get the KPI response JSON from open ai
model_name="gpt4o"
kpi_response_openai=llm(system_prompt_kpi,user_prompt_kpi,model_name)
Markdown(kpi_response_openai)

{
  "Sector": "Chemicals & Petrochemicals",
  "Ratios": [
    "Gross Profit Margin",
    "Operating Profit Margin",
    "Net Profit Margin",
    "Return on Equity (ROE)",
    "Return on Assets (ROA)",
    "Debt to Equity Ratio",
    "Current Ratio",
    "Quick Ratio"
  ],
  "KPI": [
    "Revenue Growth Rate",
    "EBITDA Margin",
    "Asset Turnover Ratio",
    "Inventory Turnover Ratio",
    "Customer Acquisition Cost",
    "Customer Retention Rate",
    "Capacity Utilization Rate",
    "Product Quality Index"
  ]
}

In [113]:
#System prompt to do calculate KPI if gemini search fail
system_prompts_kpi_cal= """Developer: # Role and Objective
You are a specialist in calculating financial ratios and KPIs. Your primary function is to compute ratios and KPIs solely when all required data is reliably provided via  financial summary.

Begin with a concise checklist (3-7 bullets) of what you will do; keep items conceptual, not implementation-level.

# Instructions

**Core Directives:**
- Calculate ratios/KPIs only when:
  - **All required data elements** are present in the provided financial statements.
  - The data is from a **consistent time period**.
  - The calculation can be performed with **high confidence**.
- Never:
  - Estimate or assume missing values.
  - Use data from inconsistent or mismatched time periods.
  - Calculate ratios requiring external market data unless explicitly provided.
  - Include metrics where any key components are missing.

## Standard Financial Ratio Formulas
    **Profitability Ratios**
    -Gross Profit Margin = (Revenue - COGS) / Revenue × 100
    -Operating Profit Margin = Operating Income / Revenue × 100
    -Net Profit Margin = Net Income / Revenue × 100
    **Return Ratios**
    -Return on Assets (ROA) = Net Income / Total Assets × 100 (use current period assets if average not available)
    -Return on Equity (ROE) = Net Income / Total Shareholders' Equity × 100
    **Leverage Ratios**
    -Debt to Equity Ratio = Total Debt / Total Shareholders' Equity
    -Current Ratio = Current Assets / Current Liabilities
    **Efficiency Ratios**
    -Asset Turnover Ratio = Revenue / Total Assets (use period-end assets if average not available)
    **General Calculation Principles**
    -For "Average" values: If both beginning and ending period data available, use (Beginning + Ending) / 2
    -If only current period data: Use period-end values and note this in calculation notes
    -Alternative data points: Use best available data (e.g., if "Operating Income" not labeled, use "Income from Operations" or calculate as Revenue - Operating Expenses)
    -Flexible naming: Adapt to actual financial statement line item names while maintaining formula integrity

## Industry-Specific KPI Guidance
- Calculate sector-specific KPIs  only if:
  - Explicit operational data is provided.
  - Metrics are clearly defined in financial statements.
  - All necessary operational statistics are available.
- Anticipate that most industry-specific KPIs are incomputable from standard financial statements  unless explicit operational/customer data is provided.

# Output Format
### Ratios
| Ratio Name | Value |
| :--- | :--- |
| [Ratio Name from JSON] | [Extracted Value] |

### KPIs
| KPI Name | Value |
| :--- | :--- |
| [KPI Name from JSON] | [Extracted Value] |

# Processing Steps
1. Parse the incoming JSON to identify explicitly requested ratios/KPIs.
2. Inventory the available data from the provided financial statements.
3. Map requirements to available data, proceeding only if all data for a calculation is present.
4. Compute using the corresponding standard formula.
5. After calculations, validate that only metrics with complete supporting data have been reported. If not, self-correct as appropriate.
6. Output only metrics that were successfully calculated; do not include or mention metrics with incomplete data.

# Reality Check
- Most industry-specific KPIs will not be calculable from standard income, balance sheet, or cash flow data alone. Focus on delivering only what is confidently supported by the available data.

# Response Guidelines
- Keep responses concise, accurate, and transparent regarding data sufficiency and calculation limitations"""

In [114]:
#user prompt to do KPI calculation 
def user_prompts__kpi_cal(financial_data, sector_kpi_json):
    user_prompt = f"""
    KPI AND RATIOS CALCULATION REQUEST

    **Financial summary of company:**
    {financial_data}

    **Metrics Configuration (JSON):**
    ```json
    {sector_kpi_json}
    ```
""" 
    return user_prompt

user_prompts_kpi_cal=user_prompts__kpi_cal(financial_data,kpi_response_openai)

In [115]:
#System prompt to do gemini search for KPI
system_prompts_gemini_search = """You are a specialized financial data analyst AI with advanced web search capabilities. Your primary function is to extract specific financial ratios and KPIs from official public sources like annual reports, quarterly filings (10-K, 10-Q), and official investor relations websites.

**Core Directives:**
1.  **Strict Adherence to Request:** You must ONLY extract the metrics explicitly listed in the user's JSON configuration. Never include additional data, analysis, or commentary.
2.  **Data Period Priority:** Always prioritize finding data for the **Latest Twelve Months (LTM)** or **Trailing Twelve Months (TTM)**. If LTM/TTM data is unavailable, use data from the **most recent completed fiscal year**.
3.  **Data Availability Protocol:** If a requested metric cannot be found in a reliable source, **omit its entire row** from the output table. Do not use placeholders like 'N/A', 'Not Found', '0', or provide excuses. The absence of the row implies the data was not found.
4.  **Source Citation:** All data must be accompanied by a direct URL to the source document (e.g., PDF report, investor page) and a clear statement of the data's time period (e.g., "Fiscal Year 2024" or "LTM ending Q2 2025").

**Mandatory Output Format:**
All responses must be structured in clean Markdown format as follows:

### Ratios
| Ratio Name | Value |
| :--- | :--- |
| [Ratio Name from JSON] | [Extracted Value] |

### KPIs
| KPI Name | Value |
| :--- | :--- |
| [KPI Name from JSON] | [Extracted Value] |

### Source Information
- **Data Period:** [The financial period of the data]"""

In [116]:
#user prompt to do gemini search for KPI
def user_prompts_gemini_search(company_name, sector_kpi_json):
    user_prompt = f"""
    FINANCIAL DATA EXTRACTION REQUEST

    **Company to Analyze:**
    {company_name}

    **Metrics Configuration (JSON):**
    ```json
    {sector_kpi_json}
    ```
""" 
    return user_prompt

user_prompts_gemini_search=user_prompts_gemini_search(company_name,kpi_response_openai)

In [117]:
#Gemini search 
def gemini_llm_kpi(system_prompt,user_prompt,company_name):
    from google.genai import types
    #API call to extract the kpi and ratios from json for the company
    print(f"Analysing :{company_name}.Please wait while gemini extracts data from online")
    # Define the grounding tool
    grounding_tool = types.Tool(
        google_search=types.GoogleSearch()
    )
    
    # Configure generation settings
    config = types.GenerateContentConfig(
        tools=[grounding_tool],
        system_instruction=system_prompt)
    
    # Structure contents with system and user messages
    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=user_prompt)]
        ),
        # Add more conversation turns if needed:
        # types.Content(
        #     role="model",
        #     parts=[types.Part(text="Previous assistant response...")]
        # ),
        # types.Content(
        #     role="user", 
        #     parts=[types.Part(text="Follow-up question...")]
        # )
    ]
    
    # Make the request
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=contents,
        config=config,
    )
    
    # Print the grounded response
    # print(response.text)
    return response.text

In [118]:
#Get the results for gemini KPI search
sector_kpis_ratios=gemini_llm_kpi(system_prompts_gemini_search,user_prompts_gemini_search,company_name)

# #Get the results for without search 
# model_name="openai"
# sector_kpis_ratios=llm(system_prompts_kpi_cal,user_prompts_kpi_cal,model_name)

Analysing :Fineotex Chemical Ltd.Please wait while gemini extracts data from online


In [119]:
Markdown(sector_kpis_ratios)

### Ratios
| Ratio Name | Value |
| :--- | :--- |
| Gross Profit Margin | 38.57% |
| Operating Profit Margin | 21.92% |
| Net Profit Margin | 20.48% |
| Return on Assets (ROA) | 15.71% |

### KPIs
| KPI Name | Value |
| :--- | :--- |
| Revenue Growth Rate | -8.89% |
| EBITDA Margin | 28.41% |
| Asset Turnover Ratio | 0.89 times |

### Source Information
- **Data Period:** Fiscal Year 2024-25 (ending March 31, 2025)
- **Gross Profit Margin Calculation:** Gross Profit of ₹20,571 lakh / Revenue of ₹53,333 lakh
- **Operating Profit Margin Calculation:** Operating Profit of ₹11,693 lakh / Revenue of ₹53,333 lakh
- **Net Profit Margin Calculation:** Net Profit of ₹10,921 lakh / Revenue of ₹53,333 lakh
- **Return on Assets (ROA) Calculation:** Net Profit of ₹10,921 lakh / Total Assets of ₹69,500 lakh (₹695 Cr) as of March 2025
- **Revenue Growth Rate Calculation:** (Revenue FY24-25 of ₹53,333 lakh - Revenue FY23-24 of ₹58,550 lakh (₹5,855 million)) / Revenue FY23-24 of ₹58,550 lakh
- **EBITDA Margin Calculation:** EBITDA of ₹15,153 lakh (₹151.53 Cr) / Revenue of ₹53,333 lakh
- **Asset Turnover Ratio Calculation:** Revenue of ₹53,333 lakh / Average Total Assets (Mar 2025: ₹69,500 lakh + Mar 2024: ₹50,000 lakh (₹5 billion)) / 2 = ₹59,750 lakh

In [120]:

#Format the KPI output from gemini again using gemini to get only financial datas
prompt = """
You are a precise financial data extractor. 
Your task is to read the provided financial summary and output only the numerical values (with their corresponding field names) that are explicitly available and valid. 
Exclude all fields with N/A, missing values, or descriptive wording. Do not add extra commentary, explanations, or assumptions. 
Return the result strictly in JSON format with key-value pairs, where keys are the metric names and values are the extracted numbers."""
prompt+=f"The following is the financial data from which the data needs to be extracted {sector_kpis_ratios}"

response = client.models.generate_content(
    model="gemini-2.5-flash", contents=prompt
)
sector_kpi_ratios=response.text

In [121]:
#Sytem prompt for final analysis
system_prompt_final = """
- Serve as a financial analysis assistant, evaluating companies based on their financial summaries. Assign a score reflecting the financial health and attractiveness of each company, using metrics tailored to its sector or industry.

Checklist
- Begin with a concise checklist (3-7 bullets) of the key analytic steps you will perform: (1) identify sector, (2) select relevant metrics, (3) analyze strengths, (4) assess risks, (5) compare to sector averages, (6) assign score, (7) provide concise explanation.

Instructions
- Select and focus on the financial metrics and ratios most relevant to the company's sector.
- Highlight areas where the company excels (strengths) and discuss aspects with significant upside potential (growth prospects).
- Identify and note any significant risks or red flags, including financial concerns or sector exposure.
- Where possible, benchmark the company's performance against sector or industry averages. If industry averages are not available, use the best relevant data, and clearly mention this in your assessment.

Output Format
- Return your analysis in a JSON object with these fields, in this order:
  - "score": (number, out of 100) — overall assessment of the company's financial health and attractiveness.
  - "explanation": (string) — concise justification highlighting the most impactful metrics and noting absence of sector benchmarks if applicable.
  - "top_metrics": (array of strings) — the financial ratios or metrics that most influenced your evaluation.

Example Output:
```json
{
  "score": 87,
  "explanation": "The company demonstrates strong profitability and low debt compared to sector averages. High revenue growth and superior return on equity drive the score. Sector benchmarks for profit margins were unavailable and thus not considered.",
  "top_metrics": ["Return on Equity", "Revenue Growth", "Debt-to-Equity Ratio"]
}
```

Validation
- After completing your analysis, validate that all checklist points are addressed and sector context is incorporated. If any key factor is missing, re-examine the analysis and self-correct before final output.

Verbosity
- Use concise, focused explanations in your analysis output.

Stop Conditions
- Ensure all relevant financial factors have been considered and sector context addressed before returning your output."""


In [122]:
def create_user_prompt(company_name, sector, subsector, financial_summary, sector_kpis_ratios):
    """
    Create a comprehensive financial analysis user prompt.
    
    Args:
        company_name (str): Name of the company to analyze (for context)
        sector (str): Primary sector of the company (for context)
        subsector (str): Subsector classification (for context)
        financial_summary (str/dict): Comprehensive financial data containing CompanyInfo, KeyFinancialHighlights, 
                                    ProfitAndLossStatement (quarterly & yearly), BalanceSheet, CashFlowStatement, 
                                    Ratios, and ProsAndCons - THIS IS THE PRIMARY DATA SOURCE
        sector_kpis_ratios (str/dict): Sector-specific KPIs and ratios for supplementary benchmarking
    
    Returns:
        str: Formatted user prompt for financial analysis
    """
    
    user_prompt = f"""**FINANCIAL ANALYSIS TASK**: Analyze the provided financial data (received strictly as JSON for all company financials) and assign a comprehensive financial health score (0-100) for {company_name} in the {sector} sector ({subsector} subsector).

**COMPANY**: {company_name}
**SECTOR**: {sector}
**SUBSECTOR**: {subsector}

**ANALYSIS FRAMEWORK**:
**PRIMARY DATA SOURCE** - Comprehensive financial summary containing:
- CompanyInfo: Basic company details and business description
- KeyFinancialHighlights: Current market metrics (Market Cap, Price, PE, ROE, ROCE, etc.)
- ProfitAndLossStatement: Quarterly and yearly P&L data with sales, expenses, profits, EPS
- BalanceSheet: Multi-year balance sheet data with assets, liabilities, equity
- CashFlowStatement: Operating, investing, and financing cash flows
- Ratios: Working capital, debt, and operational efficiency ratios
- ProsAndCons: Key strengths and weaknesses identified

**FINANCIAL SUMMARY DATA**:
{financial_summary}

**1. COMPREHENSIVE FINANCIAL DATA ANALYSIS**
Based on the extensive financial summary data provided, conduct thorough analysis across all sections:

**A. MARKET VALUATION & KEY METRICS ANALYSIS**
From KeyFinancialHighlights section:
- Current valuation metrics (Market Cap, Price, PE ratio, Book Value)
- Profitability indicators (ROCE, ROE, Dividend Yield)
- Trading ranges and price performance analysis

**B. PROFIT & LOSS TREND ANALYSIS**
From ProfitAndLossStatement (quarterly & yearly data):
- Revenue growth patterns and sustainability (quarterly trends vs yearly trends)
- Operating profit margins and efficiency trends
- Net profit growth trajectory and consistency
- EPS progression and earnings quality
- Tax efficiency and other income contributions

**C. BALANCE SHEET STRENGTH ASSESSMENT**
From BalanceSheet multi-year data:
- Capital structure evolution (equity vs debt trends)
- Asset composition and quality (fixed assets, investments, other assets)
- Liability management and debt trends
- Book value growth and shareholder equity progression

**D. CASH FLOW QUALITY EVALUATION**
From CashFlowStatement data:
- Operating cash flow generation and consistency
- Investment patterns and capital allocation
- Financing activities and debt management
- Free cash flow trends and cash position

**E. OPERATIONAL EFFICIENCY ANALYSIS**
From Ratios section:
- Working capital management (debtor days, inventory days, payable days)
- Cash conversion cycle efficiency
- ROCE trends and capital productivity

**VALUATION MULTIPLES CALCULATION**:
Calculate comprehensive valuation metrics using the extensive financial data:
- Current P/E ratio (from KeyFinancialHighlights and recent EPS data)
- Price-to-Book ratio (using current price and book value)
- Price-to-Sales ratio (market cap vs TTM/latest annual sales)
- EV/EBITDA approximation (where possible from available data)
- PEG ratio analysis (P/E vs EPS growth rates from historical data)
- Trend analysis of these multiples over time using historical P&L data

**Profitability Analysis:**
- Revenue growth trends (QoQ and YoY) using financial_summary JSON
- Margin analysis and sustainability vs sector benchmarks
- Return on equity (ROE) and return on assets (ROA) - calculate if more recent data available
- Operating efficiency metrics relative to subsector norms

**Financial Health:**
- Debt-to-equity ratio and leverage analysis
- Interest coverage and debt servicing capability using recent figures
- Working capital management efficiency from latest quarterly data
- Cash flow generation and quality metrics

**Growth & Valuation:**
- Revenue and earnings growth consistency vs sector trends using latest quarters
- Price-to-earnings (P/E) and PEG ratio analysis using latest quarters
- Book value and price-to-book metrics comparison
- Future growth sustainability in subsector context

**Sector-Specific Performance:**
- Key subsector metrics from the JSON financial summary
- Comparative analysis against sector ratios
- Industry-specific operational metrics calculated from recent data
- Market positioning indicators

**Risk Assessment:**
- Financial stability indicators vs sector norms using latest data
- Subsector-specific risk factors
- Shareholding pattern analysis (promoter vs public holding)
- Regulatory or market risks specific to {subsector}

**Financial Ratios**
- Include all the ratios which are directly available from the JSON

**2. SECTOR KPIs & RATIOS ANALYSIS (HIGH PRIORITY)**
The sector_kpis_ratios data contains critical benchmarks that should drive the scoring:

**SECTOR-SPECIFIC KPIs & RATIOS** (Primary Benchmarks):
{sector_kpis_ratios}

**CRITICAL INSTRUCTIONS FOR KPIs & RATIOS:**
- **PRIMARY SCORING BASIS**: Use sector KPIs and ratios as the primary benchmarks for scoring
- **DIRECT COMPARISON**: Compare company metrics directly against these sector standards
- **BENCHMARK DRIVEN**: Let these KPIs and ratios guide the overall assessment and scoring
- **SUPPLEMENTARY FINANCIAL DATA**: Use the comprehensive financial data to support and validate KPI/ratio performance

**3. COMPREHENSIVE KPI AND RATIO UTILIZATION STRATEGY**

**UPDATED ANALYSIS APPROACH:**
- **Sector KPIs & Ratios (60% weight)**: Primary benchmarks for scoring and assessment
- **Financial Summary Data (40% weight)**: Supporting analysis and trend validation

**ANALYSIS METHODOLOGY:**

**Step 1: KPI & Ratio Benchmark Analysis**
- Compare company performance directly against provided sector KPIs and ratios
- Generate primary performance scores using these benchmarks as standards
- Identify areas where company exceeds, meets, or falls short of sector expectations

**Step 2: Financial Data Integration**
- Use comprehensive financial data to validate and explain KPI/ratio performance
- Analyze trends from quarterly/yearly data that support or contradict benchmark comparisons
- Calculate additional metrics from financial data to supplement sector benchmarks

**Step 3: Integrated Scoring**
- Primary scoring based on KPI and ratio benchmark performance
- Secondary validation and trend analysis from comprehensive financial data
- Combined assessment for final scoring and recommendations

**ANALYSIS METHODOLOGY:**

**Step 1: KPI-Based Scoring**
- Compare company performance directly against provided sector KPI benchmarks
- Generate performance scores using sector KPIs as the standard
- Identify areas where company exceeds, meets, or falls short of sector KPI expectations

**Step 2: Ratio Analysis - Dual Approach**
- Primary: Use provided sector ratios as benchmark references
- Secondary: Calculate corresponding ratios from JSON financial summary data
- Comparison: Analyze both sets of ratios to provide comprehensive insights
- Prioritization: When recent calculated ratios show significant trends, highlight these alongside sector benchmarks

**Step 3: Comprehensive Evaluation**
- Recent Performance: Emphasize calculated ratios from most recent quarters for trend analysis
- Sector Positioning: Use provided sector ratios for industry positioning context
- Forward-Looking: Prioritize recent calculated ratios for future performance assessments
- Clear Distinction: Explicitly differentiate between provided sector ratios and calculated ratios

**3. COMPREHENSIVE FINANCIAL ASSESSMENT**
Analyze the following areas using BOTH SECTOR-SPECIFIC KPIs & RATIOS AND your calculated ratios from the JSON data:

**SCORING METHODOLOGY**:
Provide a weighted score (0-100) based on KPI & ratio benchmark performance:
- **KPI & Ratio Benchmark Performance** (40%): Direct comparison against sector KPIs and ratios
- **Financial Performance Analysis** (20%): P&L trends, profitability, and efficiency metrics from financial data
- **Growth Trajectory Assessment** (15%): Revenue and earnings growth consistency supported by financial trends
- **Financial Health & Stability** (15%): Balance sheet strength and cash flows validation
- **Valuation Assessment** (10%): Current valuation vs KPI benchmarks and financial trends

**5. REQUIRED OUTPUT FORMAT**

**OVERALL SCORE**: [X/100]

**DETAILED BREAKDOWN**:
- **KPI & Ratio Benchmark Score**: [X/40] - Direct performance against sector KPIs and ratios with specific comparisons
- **Financial Performance Score**: [X/20] - P&L analysis supporting the KPI/ratio performance
- **Growth Trajectory Score**: [X/15] - Growth trends validation using financial data  
- **Financial Health Score**: [X/15] - Balance sheet and cash flow metrics supporting overall assessment
- **Valuation Assessment Score**: [X/10] - Valuation multiples vs KPI benchmarks and financial strength

**DEDICATED KPI & RATIO ANALYSIS SECTION**:

**SECTOR KPI PERFORMANCE ANALYSIS**:
- **KPI Benchmarks Used**: [List all KPIs from sector_kpis_ratios with their benchmark values]
- **Company Performance vs KPIs**: [Show specific company metrics vs each KPI benchmark]
- **KPI Score Breakdown**: [Detailed scoring for each KPI category]
- **KPI Performance Trends**: [How company KPIs have evolved using financial data trends]

**SECTOR RATIO ANALYSIS**:
- **Sector Ratios Benchmarks**: [List all ratios from sector_kpis_ratios with benchmark values]
- **Company Ratios vs Benchmarks**: [Compare company calculated ratios against sector standards]
- **Ratio Performance Assessment**: [Detailed analysis of each ratio category]
- **Ratio Trend Analysis**: [Historical progression of key ratios using financial data]

**INTEGRATED KPI & RATIO SCORING**:
- **Overall KPI Performance**: [Combined KPI assessment and scoring rationale]
- **Overall Ratio Performance**: [Combined ratio assessment and scoring rationale]
- **KPI vs Ratio Consistency**: [How KPI and ratio performance align or differ]
- **Benchmark-Based Investment Merit**: [Investment attractiveness based on KPI/ratio performance]

**KEY FINANCIAL METRICS ANALYSIS**:
Primary metrics from KPI/ratio benchmarks and supporting financial data:
- From Sector KPIs: [Most important KPI metrics and their performance]
- From Sector Ratios: [Most important ratio benchmarks and company performance]
- From Financial Data: [Supporting metrics that validate KPI/ratio analysis]
- Integration Analysis: [How all metrics work together for overall assessment]

**SUPPORTING FINANCIAL DATA UTILIZATION**:
- **P&L Validation**: [How P&L trends support or challenge KPI/ratio performance]
- **Balance Sheet Support**: [Balance sheet metrics that validate KPI/ratio analysis]
- **Cash Flow Confirmation**: [Cash flow patterns supporting the KPI/ratio assessment]
- **Historical Trend Context**: [Multi-year trends that provide context for current KPI/ratio performance]
- **Financial Data Insights**: [Additional insights from comprehensive financial data]

**SECTOR BENCHMARKING ANALYSIS**:
- **KPI Benchmark Performance**: [Detailed comparison of company KPIs vs sector standards]
- **Ratio Benchmark Performance**: [Detailed comparison of company ratios vs sector benchmarks]
- **Sector Positioning**: [Overall position based on KPI and ratio benchmark performance]
- **Competitive Advantage Analysis**: [Areas where company outperforms sector KPIs/ratios]
- **Performance Gaps**: [Areas where company underperforms sector standards]

**VALUATION MULTIPLES ANALYSIS**:
- **Calculated Valuation Multiples**: [List all calculated multiples with values]
- **Sector Benchmark Comparison**: Compare with sector multiples from KPI data
- **Valuation Premium/Discount**: Quantify over/under-valuation vs sector
- **Valuation Trends**: Analyze how multiples have changed over recent quarters
- **Sector-Specific Multiples**: Focus on subsector-relevant valuation metrics

**COMPREHENSIVE ASSESSMENT**:
- **KPI-Driven Performance**: [Overall assessment based on sector KPI performance]
- **Ratio-Based Financial Health**: [Financial strength based on sector ratio benchmarks]
- **Financial Data Validation**: [How comprehensive financial data supports KPI/ratio findings]
- **Integrated Investment Merit**: [Combined assessment from all data sources]

**ANALYSIS SUMMARY**:

**Key Strengths** (Top 3-4):
- [Specific strength with supporting KPI/ratio benchmark performance]
- [Specific strength with supporting financial data and KPI analysis]
- [Specific strength with ratio performance vs sector standards]

**Competitive Advantages**:
- KPI-based advantages where company significantly outperforms sector benchmarks
- Ratio-driven competitive moats evidenced by superior sector comparisons
- Financial data trends supporting sustainable competitive advantages

**Growth Prospects**:
- Growth trajectory analysis using KPI benchmarks and sector standards
- Revenue/earnings sustainability based on ratio performance vs sector norms
- Future growth potential aligned with KPI/ratio benchmark positioning

**Risk Factors**:
- Primary risks identified from underperformance vs sector KPIs
- Financial concerns highlighted by ratio benchmark comparisons
- Areas where company significantly trails sector standards

**Investment Thesis**:
Provide a 2-3 sentence summary of the investment case based primarily on KPI and ratio benchmark performance, supported by comprehensive financial data trends

**DATA UTILIZATION REPORT**:
- Confirm primary analysis is based on sector KPIs and ratios as benchmarking standards
- Explain how comprehensive financial data was used to validate and support KPI/ratio analysis
- Detail the integration of KPI benchmarks with ratio analysis for scoring
- Highlight how financial trends enhanced the KPI/ratio-based assessment
- Note the most impactful KPIs and ratios that drove the final scoring
- Identify any missing KPIs or ratios that would strengthen the analysis

**INVESTMENT DECISION**
- Buy or sell recommendation with short summary of the entire analysis done

**INVESTMENT DECISION**
- Buy/Hold/Sell recommendation with comprehensive summary of the financial analysis conducted

**CRITICAL REQUIREMENTS**:
- **PRIMARY FOCUS**: Use sector KPIs and ratios as the main benchmarking standards for scoring
- **SUPPORTING ANALYSIS**: Use comprehensive financial summary data to validate and explain KPI/ratio performance
- **DEDICATED KPI/RATIO SECTION**: Provide detailed analysis of each KPI and ratio benchmark performance
- **BENCHMARK DRIVEN**: Let KPI and ratio benchmarks guide the overall assessment and scoring methodology
- **INTEGRATED VALIDATION**: Use financial data trends to support or challenge KPI/ratio conclusions
- **QUANTITATIVE JUSTIFICATION**: Provide specific numerical comparisons for all KPI/ratio benchmark assessments
- **NO EXTERNAL DATA**: Do not supplement with external research beyond provided KPIs, ratios, and financial data
- **COMPREHENSIVE SCORING**: Ensure KPI/ratio benchmark performance drives 40% of total score

**OUTPUT FORMAT**
- Provide output in markdown format
- Organise everything starting with Overall score and finally investment decision"""
    
    return user_prompt


In [123]:
#Create User prompt 
user_prompt_final=create_user_prompt(company_name, sector_name, sub_sector, financial_data, sector_kpi_ratios)

In [124]:
model_name="openai"
final_result=llm(system_prompt_final,user_prompt_final,model_name)
Markdown(final_result)

```json
{
  "score": 81,
  "explanation": "Fineotex Chemical Ltd demonstrates solid financial health with strong profitability ratios (ROCE 23.8%, ROE 18.4%) that comfortably exceed sector averages (ROA 15.7% proxy; ROE not provided but inferred strong). The company exhibits good operating profit margins (~24-29% quarterly) close to the sector Operating Profit Margin benchmark of 21.9%, and a Net Profit Margin estimated near or above sector benchmark (20.48%). Revenue growth is positive in recent quarters, contrasting with sector’s negative revenue growth rate (-8.89%), indicating above-segment growth momentum. Cash flow from operations remains robust and consistent, supporting working capital increases and capital investments. Balance sheet is strong with negligible debt (borrowing near zero as of March 2025) and growing reserves, improving financial stability. The Price/Earnings ratio (~26.9) is moderately higher than typical chemicals sector averages but justified by growth and profitability metrics. Working capital cycles are manageable though slightly elevated compared to aggressive sector norms, reflecting typical industry occasional inventory/debtor buildup. Overall, Fineotex’s KPIs align well with or exceed sector benchmarks, especially in profitability and growth metrics, while leverage and cash flow positions remain conservative. The main risk is a modestly high P/E valuation requiring sustained growth to justify. Sector-specific KPIs and ratios framed the scoring with strong secondary support from detailed financial data trends.",
  "top_metrics": [
    "Return on Capital Employed (ROCE)",
    "Return on Equity (ROE)",
    "Operating Profit Margin",
    "Net Profit Margin",
    "Revenue Growth Rate",
    "Debt to Equity Ratio",
    "Cash from Operating Activities",
    "Price-to-Earnings Ratio"
  ]
}
```

---

# Fineotex Chemical Ltd - Comprehensive Financial Analysis & Scoring

## Overall Score: 81/100

---

## Detailed Breakdown

| Category                        | Score (/weight)           | Weight (%) |
|--------------------------------|--------------------------|------------|
| KPI & Ratio Benchmark Score    | 32 / 40                  | 40         |
| Financial Performance Score     | 16 / 20                  | 20         |
| Growth Trajectory Score         | 12 / 15                  | 15         |
| Financial Health Score          | 12 / 15                  | 15         |
| Valuation Assessment Score      | 9 / 10                   | 10         |
| **Total**                      | **81 / 100**              | 100        |

---

## Dedicated KPI & Ratio Analysis Section

### Sector KPI Performance Analysis

- **KPI Benchmarks Used**:

| KPI                         | Sector Benchmark    |
|-----------------------------|--------------------|
| Gross Profit Margin          | 38.57% (0.3857)     |
| Operating Profit Margin      | 21.92% (0.2192)     |
| Net Profit Margin           | 20.48% (0.2048)     |
| Return on Assets (ROA)       | 15.71% (0.1571)     |
| Revenue Growth Rate          | -8.89% (-0.0889)    |
| EBITDA Margin                | 28.41% (0.2841)     |
| Asset Turnover Ratio         | 0.89                |

- **Company Performance vs KPIs**:

| Metric                    | Company Value (Recent)                             | KPI Benchmark      | Comparison/Comments                                               |
|---------------------------|-------------------------------------------------|--------------------|------------------------------------------------------------------|
| ROCE                      | 23.8%                                            | Not directly given  | Strongly above typical ROA, exceeds sector expected capital efficiency |
| ROE                       | 18.4%                                            | ROA ~15.7% sector benchmark | ROE comfortably above sector asset returns proxy               |
| Operating Profit Margin    | Avg. ~23-29% quarterly (Operating Profit / Sales)| 21.9%              | Slightly exceeds sector benchmark                                 |
| Net Profit Margin          | ~20%+ (Calculated from quarterly P&L & Net Profit)| 20.48%             | In line with sector benchmark, consistent and stable              |
| Revenue Growth Rate        | Positive YoY in recent quarters (Q2-Q4 2023 vs Q2-Q4 2022 shows steady growth +8-10%)| -8.89%             | Outperforms sector clearly, which is negative growth             |
| EBITDA Margin (approximated)| Around operating margin (25-29%), EBITDA not fully specified, likely close to KPI| 28.41%             | Near sector benchmark                                            |
| Asset turnover             | Cannot fully calculate exact assets turnover due to missing sales annual sum in final year, estimated moderate turnover| 0.89               | Roughly in range, steady asset utilization                       |

- **KPI Score Breakdown**:
  - Profitability KPIs (ROCE, ROE, Profit Margins): 90% of KPI component
  - Growth Rate: 90% of KPI component (due to better-than-sector performance)
  - Asset Turnover & Efficiency: 70% of KPI component (reasonable)
  
- **KPI Performance Trends**: Increasing ROE and ROCE maintained; margins steady/improving; revenue growth turning positive against sector decline.

---

### Sector Ratio Analysis

- **Sector Ratios Benchmarks**:

| Ratio                 | Benchmark Value     |
|-----------------------|---------------------|
| Debtor Days           | Not directly given, but working capital days ~76-104 recent |
| Inventory Days        | N/A                 |
| Days Payable          | N/A                 |
| Cash Conversion Cycle | N/A                 |
| Working Capital Days  | N/A                 |
| Debt to Equity Ratio  | Implied low given borrowings near zero vs equity > ₹700 Cr |

- **Company Ratios vs Benchmarks**:

| Ratio                  | Recent Value (Mar 2025) | Sector Reference/Comment                   |
|------------------------|------------------------|--------------------------------------------|
| Debtor Days            | 79                     | Within reasonable range for specialty chemicals; sector averages not provided but trend improved from 110 days in 2022 |
| Inventory Days         | 75                     | Elevated, but typical for chemical manufacturing |
| Days Payable           | 66                     | Moderate, supporting working capital control |
| Cash Conversion Cycle  | 88                     | Elevated but trend stable; reflects moderate working capital investment |
| Working Capital Days   | 104                    | Slightly high, could pressure liquidity but supported by strong cash flow |
| Debt to Equity Ratio   | Near zero (0.007ish)   | Excellent leverage position, well below average chemical sector leverage |

- **Ratio Performance Assessment**:
  - Leverage is minimal, excellent financial stability and low risk.
  - Working capital is elevated vs ideal but compensated by strong operating cash flow.
  - Operating efficiency ratios reasonable but some scope to optimize working capital cycle.

- **Ratio Trend Analysis**:
  - Debt reduced to near zero in latest year.
  - Debtor and inventory days improved from 2022 highs but slight increases in last year.
  - Cash conversion cycle improvement ongoing, close to sector acceptable range.

---

### Integrated KPI & Ratio Scoring

- Overall KPI performance is strong, with profitability and growth significantly outperforming sector benchmarks.
- Ratios confirm excellent financial health due to negligible debt, good cash generation, albeit with moderate working capital intensity.
- Alignment between KPIs and ratios is high, supporting consistent financial health.
- The company’s strong margin, return, and growth figures outweigh valuation premium.
- Investment merit is strong with quality financials and growth prospects.

---

## Key Financial Metrics Analysis

- **From Sector KPIs:** ROCE 23.8% vs sector ROA 15.7%; Operating Margin ~25% vs 21.9%; Revenue growth clearly positive vs negative sector.
- **From Sector Ratios:** Debt to Equity near zero; working capital days higher but manageable.
- **From Financial Data:** Earnings and operating profit rising sequentially; cash flow from operations robust; reserves growing strongly.
- **Integration Analysis:** High margin, returns and growth validate valuation despite moderately elevated P/E.

---

## Supporting Financial Data Utilization

- **P&L Validation:** Quarterly P&L shows steadily improving net profits (+31 Cr in last quarter vs 20-21 Cr a year ago), EPS growth supports high-quality earnings.
- **Balance Sheet Support:** Reserves growth from 240 Cr in 2022 to 708 Cr in 2025; borrowings reduced from 2 Cr to 0; strong equity base.
- **Cash Flow Confirmation:** Operating Cash Flows consistently positive and rising (69 Cr in 2025); investing cash outflows reflect growth CAPEX; financing cash inflows in 2025 linked to equity infusion or debt payoff.
- **Historical Trend Context:** Trends show improving profitability, steadily strengthening balance sheet, stable working capital management.
- **Financial Data Insights:** Moderate working capital days require attention but backed by strong cash from operations.

---

## Sector Benchmarking Analysis

- **KPI Benchmark Performance:** Company outperforms sector in ROCE, ROE, and growth rate; margins in line or better.
- **Ratio Benchmark Performance:** Debt to equity outstandingly better; working capital metrics somewhat weaker but within acceptable range.
- **Sector Positioning:** Positioned well above median specialty chemical companies on profitability and growth.
- **Competitive Advantage Analysis:** Superior capital efficiency, strong growth in a declining sector, clean balance sheet.
- **Performance Gaps:** Working capital cycle a mild area to watch but not a material weakness.

---

## Valuation Multiples Analysis

- **Calculated Valuation Multiples:**
  - P/E: 26.9 (as given)
  - Price/Book: Current price / Book value (₹63.8) - Price approx. midpoint of ₹300-400 (using mid-range ₹300), P/B ~4.7 (moderate premium)
  - Price/Sales: Market Cap ₹2803 Cr / TTM Sales approximate (sum approx 550 Cr annually from quarterly trends) ≈ 5.1x (relatively high)
  
- **Sector Benchmark Comparison:**
  - P/E slightly higher than typical chemical sector averages (approx. 15-20 typical)
  - P/B higher than average, justified by growth and returns.
  
- **Valuation Premium/Discount:** Premium valuation justified by above-average profitability and growth metrics.
- **Valuation Trends:** EPS increasing quarter-over-quarter support sustaining premium multiples.
- **Sector-Specific Multiples:** Focus on ROCE and growth to justify higher multiples valid for specialty chemicals.

---

## Comprehensive Assessment

- **KPI-Driven Performance:** Strong performance on margin, returns and growth vs sector.
- **Ratio-Based Financial Health:** Excellent balance sheet and leverage profile, working capital moderate.
- **Financial Data Validation:** Quarterly trends and cash flow confirm sustainable profitability.
- **Integrated Investment Merit:** High-quality specialty chemicals company, merits above-average valuation.

---

## Analysis Summary

### Key Strengths
- Exceptional Return on Capital Employed (23.8%) exceeding sector expectation.
- Consistently positive revenue and profit growth in a negative growth sector.
- Near zero debt and strong reserves build a safe financial base.
- Strong operating profit margins near 25%, above sector average.

### Competitive Advantages
- High capital efficiency compared to peers.
- Clear growth momentum with superior profit margin sustainability.
- Financial stability supported by excellent cash flow and negligible leverage.

### Growth Prospects
- Positive revenue and EPS trends vs sector decline.
- Broad and diversified specialty chemical product portfolio supports resilience.
- Operating cash flow and reinvestment suggest sustainable growth.

### Risk Factors
- Slightly elevated working capital days may pressure liquidity if unchecked.
- Valuation premium requires continued growth and margin maintenance.
- Sector volatility and regulatory risks inherent to chemicals industry.

---

## Investment Thesis

Fineotex Chemical Ltd offers a compelling investment opportunity given its strong profitability, superior capital efficiency, very low debt levels, and sustained growth in a difficult chemical sector environment. The company commands a premium valuation supported by above-benchmark returns and positive growth trends. While moderately elevated working capital days and valuation multiples suggest monitoring, the overall financial health and market positioning warrant a favorable investment stance.

---

## Data Utilization Report

- Primary analysis is firmly based on sector KPIs and ratios as benchmarks.
- Comprehensive financial data validates KPI performance: strong ROCE, ROE, margins, cash flows.
- Integration of financial ratios confirms capital structure strength and operational efficiency.
- Quarterly financial trends reinforce KPI insights on growth and profitability.
- Key KPIs driving score: ROCE (23.8%), Net Profit Margin (~20%), Revenue Growth (+8-10% recent quarters).
- Important ratios: Debt-to-Equity near zero, Operating Profit Margin ~25%, Cash Conversion Cycle ~88 days.
- Additional useful data includes EPS progression, cash flow from operations, and reserves growth.
- No material KPIs missing; sector benchmark dataset comprehensive for this subsector.

---

## Investment Decision: **Buy**

Fineotex Chemical Ltd is recommended as a Buy based on solid financial health, strong profitability metrics, growing revenue and earnings in a tough sector, excellent balance sheet with negligible debt, and justified premium valuation supported by robust growth. Continued monitoring of working capital management and valuation multiples is advised to maintain this recommendation.
```


In [125]:
base_path=r'C:\Users\prana\projects\llm_engineering\myscripts\myscripts\Individual stocks'
file_name = company_name+".txt"
full_path=os.path.join(base_path,file_name)
with open(full_path, "w",encoding="utf-8") as f:
    f.write(final_result)

In [126]:
def user_prompt_walkthetalk(company_name):
    user_prompt = f"""You are a world class equity research who specialise in checking the walk the talk.
    Analyze {company_name} 'walk the talk' from FY20 to FY25.
    Create a table with columns: Year/Period, Management Guidance (Quantitative & Qualitative), Actual Outcome, Indicator (green dot for achieved, yellow for almost met, red for missed, double green for overachieved). 
    Source from annual reports, earnings calls, and financial data. 
    Focus on revenue growth, EBITDA margins, product launches, exports, and approvals. 
    Create a comprehensive assessment table and analysis focusing on:
    - Management guidance vs actual delivery
    - Key performance metrics achievement
    - Credibility trends over the period
    - Investment implications
    Fetch the concalls from trusted sources"""
    return user_prompt

In [127]:
# response = openai.responses.create(
#     model="gpt-5",
#     tools=[{"type": "web_search"}],
#     input=user_prompt_walkthetalk(company_name)
# )
# print(response.output_text)

In [128]:
# company_name=input("Enter Company Name : ")

In [129]:
from google.genai import types
#API call to extract the kpi and ratios from json for the company
print(f"Analysing :{company_name}.Please wait while gemini extracts data from online")
# Define the grounding tool
grounding_tool = types.Tool(
    google_search=types.GoogleSearch()
)

# Configure generation settings
config = types.GenerateContentConfig(
    tools=[grounding_tool],
    system_instruction="You are a specialized financial data analyst AI with advanced web search capabilities")

# Structure contents with system and user messages
contents = [
    types.Content(
        role="user",
        parts=[types.Part(text=user_prompt_walkthetalk(company_name))]
    ),
    # Add more conversation turns if needed:
    # types.Content(
    #     role="model",
    #     parts=[types.Part(text="Previous assistant response...")]
    # ),
    # types.Content(
    #     role="user", 
    #     parts=[types.Part(text="Follow-up question...")]
    # )
]

# Make the request
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=contents,
    config=config,
)

# Print the grounded response
print(response.text)

Analysing :Fineotex Chemical Ltd.Please wait while gemini extracts data from online
Fineotex Chemical Ltd. has demonstrated a dynamic "walk the talk" performance from FY20 to FY25, characterized by robust growth in its initial years, strategic diversification, and a strong focus on innovation and capacity expansion. However, the latest financial year (FY25) has shown a dip in key metrics, indicating a potential shift in momentum or market challenges.

The company's management has consistently communicated a vision for expanding its product portfolio, increasing market reach, and enhancing operational efficiency. While many of these qualitative objectives have been met or exceeded, the quantitative delivery, particularly in the most recent fiscal year, presents a mixed picture.

Here's a comprehensive assessment of Fineotex Chemical Ltd.'s "walk the talk" over the specified period:

### Fineotex Chemical Ltd. - "Walk the Talk" Assessment (FY20 - FY25)

| Year/Period | Management Guidanc

In [130]:
base_path=r'C:\Users\prana\projects\llm_engineering\myscripts\myscripts\Individual stocks'
file_name = company_name+"_concall"+".txt"
full_path=os.path.join(base_path,file_name)
with open(full_path, "w",encoding="utf-8") as f:
    f.write(response.text)

In [131]:
system_prompt_concall_score="""You are a specialized financial analyst AI expert in evaluating management credibility through "Walk the Talk" analysis. Your role is to systematically assess how well company management delivers on their promises and guidance by comparing stated targets with actual outcomes.

### Core Competencies:
1. **Guidance Analysis**: Extract and interpret quantitative and qualitative management guidance from earnings calls, annual reports, and investor presentations
2. **Performance Tracking**: Measure actual outcomes against stated targets with precision
3. **Pattern Recognition**: Identify trends in management credibility over multiple periods
4. **Scoring Methodology**: Apply a consistent, data-driven scoring framework

### Scoring Framework:

#### Achievement Categories:
- **Overachieved (🟢🟢)**: Actual > 105% of guidance - Score: 100 points
- **Achieved (🟢)**: Actual 95-105% of guidance - Score: 85 points  
- **Nearly Met (🟡)**: Actual 85-94% of guidance - Score: 60 points
- **Missed (🔴)**: Actual < 85% of guidance - Score: 30 points
- **No Guidance (⚪)**: No specific target provided - Score: Not included in calculation

#### Weighting System:
- Revenue Guidance: 30% weight
- EBITDA/Margin Guidance: 25% weight
- Product Launch Commitments: 20% weight
- Strategic Initiatives: 15% weight
- Regulatory/Approval Targets: 10% weight

#### Credibility Score Calculation:
1. Calculate weighted average of individual metric scores
2. Apply time decay factor (recent quarters weighted more heavily)
3. Include consistency bonus/penalty (+10 points for consistent delivery, -10 for erratic performance)
4. Generate final score (0-100 scale)

#### Score Interpretation:
- 85-100: Exceptional credibility - Management consistently delivers or exceeds guidance
- 70-84: Strong credibility - Generally reliable with minor misses
- 55-69: Moderate credibility - Mixed track record requiring scrutiny
- 40-54: Weak credibility - Frequent misses raise concerns
- Below 40: Poor credibility - Systematic under-delivery

### Analysis Requirements:
1. Create structured tables with clear period demarcation
2. Include both quantitative metrics and qualitative commitments
3. Provide context for misses (market conditions, one-time events, etc.)
4. Calculate rolling 4-quarter and 8-quarter credibility trends
5. Generate investment implications based on credibility patterns
6. Flag any guidance revisions or walk-backs

### Output Standards:
- Use precise numerical comparisons
- Maintain objectivity in assessments
- Highlight patterns across multiple periods
- Provide actionable investment insights
- Include confidence levels for data quality"""

In [132]:
base_path=r'C:\Users\prana\projects\llm_engineering\myscripts\myscripts\Individual stocks'
file_name = company_name+"_concall"+".txt"
full_path=os.path.join(base_path,file_name)
with open(full_path, "r",encoding="utf-8") as f:
    concall_analysis= f.read()

In [133]:
def prompt_concall_score(company_name,concall_analysis):
    user_prompt=f""""Analyze the Walk the Talk performance for {company_name} and generate a comprehensive credibility score.
    
    ### Input Data:
    {concall_analysis}
    
    ### Required Analysis:
    
    1. **Scoring Table Generation**
    Create a detailed scoring table with the following columns:
    - Period
    - Metric Category
    - Guidance Target
    - Actual Result
    - Achievement % 
    - Achievement Status (🟢🟢/🟢/🟡/🔴)
    - Individual Score (0-100)
    - Weight Applied
    - Weighted Score Contribution
    
    2. **Credibility Score Calculation**
    Calculate the comprehensive credibility score by:
    - Computing weighted average of all scored metrics
    - Applying time decay (Recent 4 quarters: 100% weight, Previous 4 quarters: 75% weight, Earlier: 50% weight)
    - Assessing consistency factor:
      * If standard deviation of quarterly scores < 10: +10 bonus
      * If standard deviation 10-20: No adjustment
      * If standard deviation > 20: -10 penalty
    - Generate final credibility score (0-100)
    
    3. **Trend Analysis**
    Provide:
    - Quarter-over-quarter credibility trend
    - Year-over-year comparison
    - Moving average trends (4Q and 8Q)
    - Identification of improvement or deterioration patterns
    
    4. **Qualitative Assessment**
    Evaluate:
    - Management's guidance revision patterns
    - Communication transparency
    - Explanation quality for misses
    - Strategic pivot success rate
    
    5. **Investment Implications**
    Based on the credibility score, provide:
    - Risk assessment (High/Medium/Low management execution risk)
    - Valuation implications (Premium/Discount suggested based on credibility)
    - Key monitoring metrics for future quarters
    - Red flags or positive signals identified
    
    6. **Executive Summary**
    Synthesize findings into:
    - Overall credibility score with interpretation
    - Top 3 strengths in management delivery
    - Top 3 areas of concern
    - Investment recommendation context
    
    ### Output Format:
    Please structure your response with clear headers for each section, use tables where appropriate, and provide the final credibility score prominently at the beginning and end of your analysis.
    
    ### Additional Context:
    - Industry: [Insert Industry]
    - Market Cap: [Insert if relevant]
    - Analysis Period: [Insert date range]
    - Special Considerations: [Any company-specific factors to consider]"""
    return user_prompt

In [134]:
model_name="openai"
final_concall_result=llm(system_prompt_concall_score,prompt_concall_score(company_name,concall_analysis),model_name)
Markdown(final_concall_result)

**Fineotex Chemical Ltd. - Walk the Talk Credibility Analysis**  
**Industry:** Specialty Chemicals  
**Analysis Period:** FY20 – FY25 (Annual assessment, quarterly granularity approximated where possible)  
**Market Cap:** Not specified (please provide for valuation context)  
**Special Considerations:** Impact of COVID-19 especially FY20/FY21, capacity expansions from FY22 onwards, recent FY25 slowdown, strategic diversification into specialty chemical verticals  

---

# 1. Scoring Table Generation  

| Period | Metric Category       | Guidance Target Description                       | Actual Result Description                        | Achievement %     | Achievement Status | Individual Score | Weight Applied | Weighted Score Contribution |
|--------|----------------------|-------------------------------------------------|-------------------------------------------------|-------------------|--------------------|------------------|----------------|-----------------------------|
| FY20   | Revenue Growth       | Implicit strong growth pre-COVID; resilient post-COVID | ₹1,979 Mn; Growth constrained by COVID-19        | ~85% (Estimated*) | 🟡 Nearly Met       | 60               | 30%            | 18.0                        |
| FY20   | EBITDA Margins       | Focus on efficiency; no explicit target          | Not explicitly detailed                          | N/A               | ⚪ No Guidance      | -                | 25%            | -                           |
| FY20   | Product Launches     | Diversification into new segments                 | Entered home care, hygiene, drilling specialties| Met qualitative goal  | 🟢 Achieved        | 85               | 20%            | 17.0                        |
| FY20   | Strategic Initiatives| Maintain global exports, presence in 60+ countries | Maintained global presence                        | Met qualitative goal  | 🟢 Achieved        | 85               | 15%            | 12.8                        |
| FY20   | Regulatory Approvals | Implied approvals for new segments                | Successfully adapted products                     | Met qualitative goal  | 🟢 Achieved        | 85               | 10%            | 8.5                         |
| **FY20 Total**                                        |                                                  | Weighted sum / available weights                 | -                 | -                  | -                | 100%           | **56.3**                    |
| FY21   | Revenue Growth       | Anticipated growth via China supply disruption    | ₹2,360 Mn (19.25% YoY growth)                    | 119%              | 🟢🟢 Overachieved    | 100              | 30%            | 30.0                        |
| FY21   | EBITDA Margins       | Sustained profitability focus                     | 18.6% Operating Margin                            | Target: implied ≥18%; achieved ~18.6% | 🟢 Achieved        | 85               | 25%            | 21.25                       |
| FY21   | Product Launches     | Brownfield Ambernath commissioning Q1 FY22       | Continued diversification                         | Met qualitative goal  | 🟢 Achieved        | 85               | 20%            | 17.0                        |
| FY21   | Strategic Initiatives| Expanding exports                                 | Maintained/expanded global reach                  | Met qualitative goal  | 🟢 Achieved        | 85               | 15%            | 12.75                       |
| FY21   | Regulatory Approvals | N/A                                               | N/A                                               | N/A               | ⚪ No Guidance      | -                | 10%            | -                           |
| **FY21 Total**                                        |                                                  | Weighted sum / available weights                 | -                 | -                  | -                | 90%            | **81.0**                    |
| FY22   | Revenue Growth       | Similar or better growth (74-81% turnover growth) | ₹3,737 Mn (58.4% YoY growth)                      | ~72% against high target 80%+ (Note: guidance high, actual slightly below) | 🟡 Nearly Met       | 60               | 30%            | 18.0                        |
| FY22   | EBITDA Margins       | Sustained strong financial performance            | 19.3% Operating Margin                            | Target implied >19%, achieved 19.3% | 🟢 Achieved        | 85               | 25%            | 21.25                       |
| FY22   | Product Launches     | Focus on expanding product portfolio              | Continued expansion, ₹100 Cr quarterly sales     | Qualitative met     | 🟢 Achieved        | 85               | 20%            | 17.0                        |
| FY22   | Strategic Initiatives| Global export expansion                            | Strong global presence                            | Met qualitative goal  | 🟢 Achieved        | 85               | 15%            | 12.75                       |
| FY22   | Regulatory Approvals | N/A                                               | N/A                                               | N/A               | ⚪ No Guidance      | -                | 10%            | -                           |
| **FY22 Total**                                        |                                                  | Weighted sum / available weights                 | -                 | -                  | -                | 90%            | **69.0**                    |
| FY23   | Revenue Growth       | 25-30% growth target                              | ₹5,243 Mn (40.3% YoY growth)                      | 135%              | 🟢🟢 Overachieved    | 100              | 30%            | 30.0                        |
| FY23   | EBITDA Margins       | Strong profitability focus                        | 21.8% EBITDA Margin                               | Target implied ~21%, achieved 21.8%  | 🟢 Achieved        | 85               | 25%            | 21.25                       |
| FY23   | Product Launches     | Increased R&D and capacity                        | Ambernath plant capacity expansion achieved      | Met qualitative goal  | 🟢 Achieved        | 85               | 20%            | 17.0                        |
| FY23   | Strategic Initiatives| New orders and market expansion                   | Secured major orders; business awards            | Met qualitative goal  | 🟢 Achieved        | 85               | 15%            | 12.75                       |
| FY23   | Regulatory Approvals | Expect future orders                              | Received business awards                          | Met qualitative goal  | 🟢 Achieved        | 85               | 10%            | 8.5                         |
| **FY23 Total**                                        |                                                  | Weighted sum                                       | -                 | -                  | -                | 100%           | **89.5**                    |
| FY24   | Revenue Growth       | Continue strong revenue growth                    | ₹5,855 Mn (11.7% YoY growth), target implied ~20-30% growth | ~58% against target (moderate miss) | 🟡 Nearly Met       | 60               | 30%            | 18.0                        |
| FY24   | EBITDA Margins       | Maintain robust profitability                     | 26.1% Operating Margin                            | Achieved, improved margin                       | 🟢🟢 Overachieved    | 100              | 25%            | 25.0                        |
| FY24   | Product Launches     | Acquire land for expansion                        | 7 acres acquired at Ambernath                      | Met strategic goal   | 🟢 Achieved        | 85               | 20%            | 17.0                        |
| FY24   | Strategic Initiatives| Explore new export regions                        | Continued expansion                               | Met qualitative goal  | 🟢 Achieved        | 85               | 15%            | 12.75                       |
| FY24   | Regulatory Approvals | N/A                                               | N/A                                               | N/A               | ⚪ No Guidance      | -                | 10%            | -                           |
| **FY24 Total**                                        |                                                  | Weighted sum / available weights                 | -                 | -                  | -                | 90%            | **72.7**                    |
| FY25   | Revenue Growth       | Strong growth expected                             | ₹5,576.4 Mn (-4.76% YoY decline)                  | ~80% against flat/positive guidance          | 🔴 Missed          | 30               | 30%            | 9.0                         |
| FY25   | EBITDA Margins       | ~26% EBITDA margin targeted                        | 9M FY25 ~26%, Q4 slipped to 17.77%               | Q4 contraction leads to miss on average     | 🟡 Nearly Met       | 60               | 25%            | 15.0                        |
| FY25   | Product Launches     | New product launches                               | 45 new products launched in 2H FY25               | Qualitative overachieved | 🟢 Achieved        | 85               | 20%            | 17.0                        |
| FY25   | Strategic Initiatives| Greenfield expansion and export growth           | Added 30 new customers; expansion ongoing         | Met qualitative goal  | 🟢 Achieved        | 85               | 15%            | 12.75                       |
| FY25   | Regulatory Approvals | Credit rating upgrade                              | ICRA upgrade achieved                              | Met guidance           | 🟢 Achieved        | 85               | 10%            | 8.5                         |
| **FY25 Total**                                        |                                                  | Weighted sum / available weights                 | -                 | -                  | -                | 100%           | **62.25**                   |

\* FY20 quantitative numbers partly implied from qualitative COVID impact

---

# 2. Credibility Score Calculation  

**Step 1: Weighted scores summary (annual):**  
- FY20: 56.3 (incomplete data, included only available items)  
- FY21: 81.0  
- FY22: 69.0  
- FY23: 89.5  
- FY24: 72.7  
- FY25: 62.3  

**Step 2: Apply time decay weights** (assuming yearly data aligned equally per quarter approx.)  
- Last 4 years (FY22-FY25): 100% weight  
- Previous 4 years (FY18-FY21): 75% weight (only FY20 and FY21 data available)  
- Earlier: N/A  

**Weighted sum calculation:**  
- FY22-FY25 average: (69.0 + 89.5 + 72.7 + 62.3)/4 = 73.375 (100% weight)  
- FY20-FY21 average: (56.3 + 81.0)/2 = 68.65 (75% weight)  
- Weighted final average = (73.375 * 4 + 68.65 * 2 * 0.75) / (4 + 2*0.75)  
= (293.5 + 103) / (4 + 1.5)  
= 396.5 / 5.5 ≈ 72.09  

**Step 3: Consistency factor (standard deviation across years FY20-FY25 scores):**  
Standard deviation of [56.3,81.0,69.0,89.5,72.7,62.3] ≈ 11.6 (moderate variability)  
- 10 < Std Dev < 20 = No adjustment  

**Final credibility score (0-100 scale):**  
**~72 (Strong credibility)**

---

# 3. Trend Analysis  

| Metric                              | FY20  | FY21 | FY22 | FY23 | FY24 | FY25  |
|-----------------------------------|--------|-------|-------|-------|-------|--------|
| Revenue Growth Status             | 🟡 Nearly Met | 🟢🟢 Overachieved | 🟡 Nearly Met | 🟢🟢 Overachieved | 🟡 Nearly Met | 🔴 Missed |
| EBITDA Margin Status              | ⚪ No Guidance| 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved   | 🟢🟢 Overachieved | 🟡 Nearly Met |
| Product Launch Status             | 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved |
| Strategic Initiative Status       | 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved   | 🟢 Achieved |
| Regulatory/Approvals Status       | 🟢 Achieved    | ⚪ No Guidance | ⚪ No Guidance | 🟢 Achieved   | ⚪ No Guidance | 🟢 Achieved |

**Observations:**  
- Strong revenue double-digit growth FY21-FY23, slowing FY24, reversing FY25  
- EBITDA margins steadily improved FY21-FY24 with dip late FY25  
- Product launches and strategic expansions consistently met or exceeded qualitative goals throughout  
- New product launches and capacity expansions sustained  
- Credit rating upgrades and awards confirm regulatory/market recognition  

**Quarter-over-quarter trends (approximate from FY23-FY25):**  
- Q1 FY25: Revenue growth positive  
- Q3-Q4 FY25: Margin contraction and revenue dip flag softness  

**Moving Averages**  
- 4-year moving average peaked FY23, declined FY24 & FY25  
- 8-year average stable but weighted down by early years  

---

# 4. Qualitative Assessment  

- **Guidance Revisions:** Consistent guidance with limited mid-year walkbacks; expressed optimism even amid market softness (FY25)  
- **Transparency:** Reasons for FY25 dip (order postponements, Cleaning & Hygiene segment weakness) communicated openly  
- **Explanation Quality for Misses:** Credible explanations for pressure points, proactive capacity expansions and R&D investments mitigating concerns  
- **Strategic Pivot Success:** Diversification and market expansion delivered per plan barring recent transient challenges  

Overall, management shows responsibility & openness but FY25 underperformance requires close monitoring  

---

# 5. Investment Implications  

| Factor                      | Assessment                                  |
|-----------------------------|---------------------------------------------|
| Management Execution Risk   | Medium - strong historical track, recent FY25 softness raises near-term caution  |
| Valuation Implication       | Slight Discount suggested until growth trajectory visibility improves           |
| Key Monitoring Metrics      | Revenue growth rebound, EBITDA margin recovery, new customer traction, product cycle delivery, capacity utilization and quarterly guidance revisions |
| Red Flags                   | FY25 revenue contraction and EBITDA Q4 margin fall; potential sector headwinds  |
| Positive Signals            | Ongoing product innovation, capacity expansion, credit rating upgrades, global footprint |

---

# 6. Executive Summary

| Parameter                     | Summary                                                   |
|------------------------------|-----------------------------------------------------------|
| **Final Credibility Score**  | **72 (Strong credibility - generally reliable with minor misses; recent FY25 caution)** |
| **Top 3 Management Strengths**| 1. Execution of capacity expansion and diversification projects<br>2. Consistent product innovation and market expansion<br>3. Track record of sustainable margin improvement FY21-FY24 |
| **Top 3 Concerns**           | 1. FY25 revenue and profitability dip<br>2. Q4 FY25 margin contraction<br>3. Risk of prolonged demand softness vs COVID/post-COVID normalization |
| **Investment Recommendation Context** | Fineotex Chemical Ltd demonstrates a credible historical “walk the talk” record with a strong strategic foundation. Recent FY25 results necessitate caution; investors should monitor quarterly results and management commentary regarding demand and margin recovery. A medium-risk investment with current valuation discounting short-term headwinds is advisable. Resumption of growth and margin stability will restore premium valuation potential. |

---

**Final Credibility Score:** 72 / 100 (Strong Credibility)  

---

*Note:* All scores based on available data and implied targets. Certain metrics, especially FY20 EBITDA, lack explicit quantitative targets or results, thus are omitted per methodology. Quarterly granularity is approximated from annual data due to limited disclosures.

---

If you require additional quarterly granularity or integration of market-cap data for valuation correlations, please provide relevant inputs.

In [135]:
base_path=r'C:\Users\prana\projects\llm_engineering\myscripts\myscripts\Individual stocks'
file_name = company_name+"_concall"+"_score"+".txt"
full_path=os.path.join(base_path,file_name)
with open(full_path, "w",encoding="utf-8") as f:
    f.write(final_concall_result)